In [1]:
##### Pull the latest version of nano-graphrag from the github page
!pip uninstall -y nano-graphrag nano_graphrag
!git clone https://github.com/gusye1234/nano-graphrag.git
%cd nano-graphrag
!pip install -e .

import nano_graphrag
nano_graphrag.__version__

Found existing installation: nano-graphrag 0.0.8.2
Uninstalling nano-graphrag-0.0.8.2:
  Successfully uninstalled nano-graphrag-0.0.8.2
fatal: destination path 'nano-graphrag' already exists and is not an empty directory.
/content/nano-graphrag
Obtaining file:///content/nano-graphrag
  Preparing metadata (setup.py) ... done
  Running setup.py develop for nano-graphrag


'0.0.8.2'

In [2]:
##### Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
##### Install all the necessary libraries
!pip install numpy ollama nest-asyncio networkx sentence-transformers transformers --quiet

##### Start Ollama server
import subprocess
import time
import os

subprocess.run(['pkill', '-f', 'ollama'], stderr=subprocess.DEVNULL)
time.sleep(2)

os.environ['OLLAMA_KEEP_ALIVE'] = '-1'

subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL,
                 stderr=subprocess.DEVNULL)
time.sleep(5)

subprocess.run(['ollama', 'pull', 'nomic-embed-text'])
subprocess.run(['ollama', 'pull', 'qwen2'])
# import time
# !pkill -f ollama  # kill leftover Ollama processes
# !ollama serve > /dev/null 2>&1 &  # run in background quietly
# time.sleep(5)  # wait for server to initialize

# ##### Pull required models
# !ollama pull nomic-embed-text
# !ollama pull qwen2

CompletedProcess(args=['ollama', 'pull', 'qwen2'], returncode=0)

In [4]:
##### Edit Modelfile to upgrade the llm from qwen2 to one with a bigger context size
# !ollama show --modelfile qwen2 > Modelfile
# In the modelfile created, add a new line into this file below the 'FROM':
# PARAMETER num_ctx 32000

In [5]:
# run this line or add Modelfile into /content/nano-graphrag/Modelfile
# !ollama create -f Modelfile qwen2:ctx32k

In [6]:
##### Our input documents
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# Paths and working directories
# -------------------------------
DATA_DIR = "/content/drive/MyDrive/erica/data"
WORKING_DIR = "/content/drive/MyDrive/erica/nano_graphrag_cache_ollama"
BACKUP_DIR = "/content/drive/MyDrive/erica/nano_graphrag_cache_ollama/backup"
# MODEL = "qwen2"
MODEL = "qwen2:ctx32k"
BATCH_SIZE = 10  # save progress every 10 pages
os_mkdirs = True

# Make sure working dir exists
import os
if not os.path.exists(WORKING_DIR):
    os.makedirs(WORKING_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# #### Inspect our documents
# import gzip
# import json
# from pathlib import Path

# for file in Path(DATA_DIR).glob("*.json.gz"):
#     print(f"\n============================")
#     print(f"FILE: {file.name}")
#     print("============================")

#     # 1. Show first few raw bytes
#     with open(file, "rb") as f:
#         raw_bytes = f.read(200)
#     print("\n Raw .gz bytes (first 200):")
#     print(raw_bytes)

#     # 2. Show decompressed text (first ~500 chars)
#     try:
#         with gzip.open(file, "rt", encoding="utf-8") as f:
#             decompressed_text = f.read()
#     except Exception as e:
#         print(f"Failed to decompress: {e}")
#         continue

#     print("\n Decompressed text preview (first 500 chars):")
#     print(decompressed_text[:500])

#     # 3. Parse JSON and show structure
#     try:
#         obj = json.loads(decompressed_text)
#     except Exception as e:
#         print(f"Failed to parse JSON: {e}")
#         continue

#     print("\n Parsed JSON type:", type(obj))

#     # If it's a list → treat as pages
#     if isinstance(obj, list):
#         print(f" Number of items in list: {len(obj)}")

#         # Show 1st item structure
#         if len(obj) > 0:
#             print("\n First item keys:", list(obj[0].keys()))
#             print(" First item preview:")
#             print(json.dumps(obj[0], indent=2)[:500])

#     # If it's a dict → maybe a single page
#     elif isinstance(obj, dict):
#         print(" Dict keys:", list(obj.keys()))
#         print("\n Dict preview:")
#         print(json.dumps(obj, indent=2)[:500])

#     print("\n--- END OF FILE INSPECTION ---\n")

In [ ]:
# ===============================
# Data Ingestion: build / resume RAG graph
# ===============================
# -------------------------------
# Imports
# -------------------------------
import os, gzip, json, re
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()
import subprocess
import time
import logging
import ollama
import torch
import numpy as np
from nano_graphrag import GraphRAG, QueryParam
from nano_graphrag._utils import wrap_embedding_func_with_attrs, compute_args_hash
from nano_graphrag.base import BaseKVStorage
from sentence_transformers import SentenceTransformer
import nano_graphrag._utils # Import here to ensure it's available for patching
import shutil

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
logger = logging.getLogger("nano-graphrag")

# -------------------------------
# Patch nano_graphrag._utils.load_json
# -------------------------------
def _load_json_patched(file_name):
    """Loads JSON from a file, handles FileNotFoundError and JSONDecodeError."""
    if not os.path.exists(file_name):
        return None
    try:
        with open(file_name, encoding="utf-8") as f:
            content = f.read().strip()
            if not content: # Handle empty file explicitly
                logger.info(f"File {file_name} is empty. Returning empty dictionary.")
                return {} # Return empty dict for empty files
            return json.loads(content)
    except json.JSONDecodeError as e:
        logger.warning(f"JSONDecodeError when loading {file_name}: {e}. Returning empty dict.")
        return {} # Return empty dict on decode error
    except Exception as e:
        logger.warning(f"Unexpected error when loading {file_name}: {e}. Returning None.")
        return None

nano_graphrag._utils.load_json = _load_json_patched
print("Patched nano_graphrag._utils.load_json to handle empty/invalid JSON files.")

# -------------------------------
# Device auto-detect
# -------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# -------------------------------
# Embedding model
# -------------------------------
EMBED_MODEL = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    cache_folder=WORKING_DIR,
    device=device
)

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBED_MODEL.get_sentence_embedding_dimension(),
    max_token_size=EMBED_MODEL.max_seq_length,
)
async def local_embedding(texts: list[str]):
    return EMBED_MODEL.encode(texts, normalize_embeddings=True)

# -------------------------------
# Ollama wrapper
# -------------------------------
# JSON helper: aggressive extraction + repair
def try_parse_json(raw_text: str):
    """
    Aggressive JSON extraction + attempt common repairs.
    Returns a JSON-string (serialized JSON). Always returns some JSON string.
    If irreparably broken, returns a minimal safe JSON: {"nodes": [], "edges": []}
    """
    if raw_text is None:
        return json.dumps({"nodes": [], "edges": []})

    s = raw_text.strip()

    # Normalize smart quotes → straight quotes
    s = s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2019", "'").replace("\u2018", "'")

    # Remove markdown fences/backticks often added by LLMs
    s = re.sub(r"```(?:json)?", "", s)
    s = s.replace("```", "")

    # If it's already valid JSON, return canonical dump
    try:
        parsed = json.loads(s)
        return json.dumps(parsed)
    except Exception:
        pass

    # Try to extract the largest { ... } block
    # Find first "{" and last "}" and slice
    first = s.find("{")
    last = s.rfind("}")
    if first != -1 and last != -1 and last > first:
        candidate = s[first:last+1]
    else:
        candidate = s

    # Common minor repairs
    # 1) Replace single quotes with double quotes where safe (very naive)
    #    Only do this if double quotes appear fewer times than single quotes to avoid breaking legitimate JSON
    if candidate.count('"') < candidate.count("'"):
        candidate = candidate.replace("'", '"')

    # 2) Remove trailing commas before } or ]
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)

    # 3) Ensure boolean/null are lowercase (if model printed True/False/None)
    candidate = candidate.replace("True", "true").replace("False", "false").replace("None", "null")

    # Attempt parse; if fails, try minor character fixes
    try:
        parsed = json.loads(candidate)
        return json.dumps(parsed)
    except Exception:
        pass

    # As a last resort, attempt to extract multiple {...} blocks and pick the largest valid one
    matches = re.findall(r"\{(?:[^{}]|\n|\r|\t)*\}", s)
    best = None
    for m in matches:
        try:
            parsed = json.loads(m)
            # pick the largest valid JSON object
            if best is None or len(m) > len(best):
                best = m
        except Exception:
            continue
    if best is not None:
        try:
            parsed = json.loads(best)
            return json.dumps(parsed)
        except Exception:
            pass

    # Give up -> return safe empty graph JSON
    logger.warning("Could not repair JSON from model output. Returning safe empty graph.")
    return json.dumps({"nodes": [], "edges": []})

async def ollama_model_if_cache(
    prompt: str,
    system_prompt: str = None,
    history_messages=None,
    **kwargs
):
    import json
    import re
    import ollama
    print("WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~")

    if history_messages is None:
        history_messages = []

    # Remove arguments Ollama does not support
    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    if system_prompt is None:
        system_prompt = ""

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    # print("======== ENTER ollama_model_if_cache ========")
    # print("MODEL:", MODEL)
    # print("RAW PROMPT:\n", prompt)
    # print("SYSTEM PROMPT:\n", system_prompt)
    # print("HISTORY MESSAGES:\n", history_messages)
    # print("================================================")

    # ---- Caching ----
    hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
    args_hash = None
    if hashing_kv:
        args_hash = compute_args_hash(MODEL, messages)
        cached = await hashing_kv.get_by_id(args_hash)
        if cached is not None:
            return cached["return"]

    # ---- Call Ollama ----
    # print("\n---- MESSAGES SENT TO OLLAMA ----")
    # print(json.dumps(messages, indent=2))
    # print("Calling model:", MODEL)
    # print("---------------------------------\n")

    ollama_client = ollama.AsyncClient()
    try:
        response = await ollama_client.chat(
            model=MODEL,
            messages=messages,
            # format = "json",
            **kwargs)
        # Safely access 'message' and 'content' to avoid KeyError
        raw = response.get("message", {}).get("content", "")
        if not raw:
            logger.warning("Ollama returned an empty or malformed message content.")
            raw = '{"nodes": [], "edges": []}' # Fallback if content is missing
        print(f"llm output is {raw}")
    except Exception as e:
        logger.error(f"Ollama call failed: {e}")
        print(f"OLLAMA CALL FAILED~~~~~~~~~~~~~~~~~~~~~~~~~ {e}")

        restart_ollama()
        raw = '{"nodes": [], "edges": []}'  # safe fallback

    return raw

# -------------------------------
# Utility
# -------------------------------
def remove_if_exist(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)

def restart_ollama():
    """Restart the Ollama server and recreate the custom model"""
    print("Restarting Ollama server...")
    try:
        # Kill existing Ollama process
        subprocess.run(["pkill", "ollama"], check=False)
        time.sleep(2)

        # Start Ollama server
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)

        # Recreate the custom model from Modelfile
        print("Recreating custom model from Modelfile...")
        result = subprocess.run(
            ["ollama", "create", "-f", "Modelfile", "qwen2:ctx32k"],
            capture_output=True,
            text=True
        )

        if result.returncode == 0:
            print("Custom model qwen2:ctx32k created successfully")
        else:
            print(f"Warning: Model creation returned code {result.returncode}")
            print(f"stdout: {result.stdout}")
            print(f"stderr: {result.stderr}")

        time.sleep(2)
        print("Ollama server restarted with custom model")

    except Exception as e:
        print(f"Error restarting Ollama: {e}")

def load_json(file_name):
    """Loads JSON from a file, handles FileNotFoundError and JSONDecodeError."""
    # This function is now effectively overridden by _load_json_patched in the global scope
    # but we keep it here for completeness and if it's called directly by non-patched code.
    if not os.path.exists(file_name):
        return None
    try:
        with open(file_name, encoding="utf-8") as f:
            content = f.read().strip()
            if not content:
                logger.info(f"File {file_name} is empty. Returning empty dictionary.")
                return {}
            return json.loads(content)
    except json.JSONDecodeError as e:
        logger.warning(f"JSONDecodeError when loading {file_name}: {e}. Returning None.")
        return None
    except Exception as e:
        logger.warning(f"Unexpected error when loading {file_name}: {e}. Returning None.")
        return None

class BackedUpGraphRAG(GraphRAG):
    def __init__(self, working_dir, backup_dir="backup", *args, **kwargs):
        self._backup_working_dir = Path(working_dir)  # private attribute for backup
        self.backup_dir = Path(backup_dir)
        super().__init__(working_dir=working_dir, *args, **kwargs)

    def _create_backup(self):
        """Replace the backup directory with current working directory contents."""
        wd = self._backup_working_dir
        if not wd.exists() or not any(wd.iterdir()):
            return

        # Remove old backup if it exists
        if self.backup_dir.exists():
            shutil.rmtree(self.backup_dir)

        # Create new backup
        shutil.copytree(wd, self.backup_dir)
        print(f"Backup updated: {self.backup_dir}")

    def insert(self, *args, **kwargs):
        """Backup before inserting new data."""
        self._create_backup()
        return super().insert(*args, **kwargs)

    def query(self, *args, **kwargs):
        """Query doesn't modify state, no backup needed."""
        return super().query(*args, **kwargs)


# -------------------------------
# Insert function: resume-safe
# -------------------------------
def insert(max_items_per_file=None, start_from_page=0):
    """
    Index pages from JSON.GZ files safely.
    Skips pages that raise errors (e.g., invalid JSON).
    Progress is stored in WORKING_DIR/progress.json for resuming.
    """
    progress_file = Path(WORKING_DIR) / "progress.json"

    # Remove previous cache if starting fresh
    if start_from_page == 0:
        for f in [
            "vdb_entities.json",
            "kv_store_full_docs.json",
            "kv_store_text_chunks.json",
            "kv_store_community_reports.json",
            "graph_chunk_entity_relation.graphml",
        ]:
            remove_if_exist(Path(WORKING_DIR) / f)

    # print("WE ARE INSIDE THE INSERT FTN~~~~~~~~~~~~~~~~~~~~~")
    # rag = GraphRAG(
    #     working_dir=WORKING_DIR,
    #     enable_llm_cache=True,  # 0.0.8 uses this flag
    #     best_model_func=ollama_model_if_cache,
    #     cheap_model_func=ollama_model_if_cache,
    #     embedding_func=local_embedding,
    # )
    rag = BackedUpGraphRAG(
        working_dir=WORKING_DIR,
        backup_dir=BACKUP_DIR,
        enable_llm_cache=True,
        best_model_func=ollama_model_if_cache,
        cheap_model_func=ollama_model_if_cache,
        embedding_func=local_embedding,
    )


    print("Using model:", MODEL)
    resp = subprocess.run(["ollama", "show", MODEL], capture_output=True, text=True)
    print(resp.stdout)


    page_counter = start_from_page
    print(f"starting at {page_counter}~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

    for file in Path(DATA_DIR).glob("*.json.gz"):
        print(f"Loading {file}")
        try:
            with gzip.open(file, "rt", encoding="utf-8") as f:
                obj = json.load(f)
        except Exception as e:
            print(f"Skipping {file}: failed to load JSON - {e}")
            continue

        pages = obj if isinstance(obj, list) else [obj]
        pages_to_process = pages if max_items_per_file is None else pages[:max_items_per_file]

        for i, page in enumerate(pages_to_process):
            if i < start_from_page:
                page_counter += 1
                print(f"Skipping page {i} in {file.name}: already processed")
                continue

            text = page.get("text", "").strip()
            if not text:
                print(f"Skipped page {i} in {file.name}: no text")
                page_counter += 1
                continue

            print(f"Inserting page {i} from {file.name} | length: {len(text)}")
            print("Chunk length:", len(text.split()))
            if len(text.split()) > 10000:
                print(f"Skipping page {i} in {file.name}: too long")
                page_counter += 1
                continue

            try:
                rag.insert(text)
            except Exception as e:
                print(f"Failed to insert page {i}: {e}")
                import traceback
                traceback.print_exc()
                # skip this page and continue
                page_counter += 1
                # update progress so we don't retry the same failing page
                progress_file.write_text(json.dumps({"last_page": page_counter}))
                continue

            page_counter += 1
            # Save progress every page
            progress_file.write_text(json.dumps({"last_page": page_counter}))

            if page_counter % BATCH_SIZE == 0:
                print(f"--- Reached {page_counter} pages ---")
                restart_ollama()

    print(f"--- Finished inserting {page_counter} pages ---")

Patched nano_graphrag._utils.load_json to handle empty/invalid JSON files.
Using device: cuda


In [9]:
##### Run this code and Restart and Run all if the next block keeps running into JSONDecodeError: Expecting value: line 1 column 1 (char 0)
# Check the content of the _utils.py file to verify the patch
#!cat /content/nano-graphrag/nano_graphrag/_utils.py

In [ ]:
import json
from pathlib import Path
import os # Import os module to handle file operations

progress_file = Path(WORKING_DIR) / "progress.json"
start_from = 0
if progress_file.exists():
    start_from = json.load(open(progress_file))["last_page"]

insert(max_items_per_file=None, start_from_page=start_from)  # process all remaining pages

INFO:nano-graphrag:Loading tokenizer: type='tiktoken', name='gpt-4o'
INFO:nano-graphrag:Load KV full_docs with 53 data
INFO:nano-graphrag:Load KV text_chunks with 150 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 20 data
INFO:nano-graphrag:Loaded graph from /content/drive/MyDrive/erica/nano_graphrag_cache_ollama/graph_chunk_entity_relation.graphml with 1145 nodes, 513 edges


Using model: qwen2:ctx32k

starting at 61~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Loading /content/drive/MyDrive/erica/data/scraped_pages.json.gz
Skipping page 0 in scraped_pages.json.gz: already processed
Skipping page 1 in scraped_pages.json.gz: already processed
Skipping page 2 in scraped_pages.json.gz: already processed
Skipping page 3 in scraped_pages.json.gz: already processed
Skipping page 4 in scraped_pages.json.gz: already processed
Skipping page 5 in scraped_pages.json.gz: already processed
Skipping page 6 in scraped_pages.json.gz: already processed
Skipping page 7 in scraped_pages.json.gz: already processed
Skipping page 8 in scraped_pages.json.gz: already processed
Skipping page 9 in scraped_pages.json.gz: already processed
Skipping page 10 in scraped_pages.json.gz: already processed
Skipping page 11 in scraped_pages.json.gz: already processed
Skipping page 12 in scraped_pages.json.gz: already processed
Skipping page 13 in scraped_pages.json.gz: already processed
Skipping pa

INFO:nano-graphrag:[New Docs] inserting 1 docs
INFO:nano-graphrag:[New Chunks] inserting 1 chunks
INFO:nano-graphrag:[Entity Extraction]...
ERROR:nano-graphrag:Ollama call failed: model 'qwen2:ctx32k' not found (status code: 404)


Backup updated: /content/drive/MyDrive/erica/nano_graphrag_cache_ollama/backup
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
OLLAMA CALL FAILED~~~~~~~~~~~~~~~~~~~~~~~~~ model 'qwen2:ctx32k' not found (status code: 404)
Restarting Ollama server...
Recreating custom model from Modelfile...
Custom model qwen2:ctx32k created successfully
Ollama server restarted with custom model
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
llm output is ("entity"<|>"organization"<|>"VAE Architecture"<|>"An architecture related to VAE that facilitates defining latent spaces for probabilistic graphical models.")## 
("entity"<|>"person"<|>"V": The Expectation - Maximization (EM) Algorithm is associated with the 'V' class, implying it's a class name or acronym related to a person involved in this algorithm.)## 
("entity"<|>"geo"<|>"Summer 2025": This seems to refer to a geographical period when notes are scheduled to be available during summer of year 2025.)

INFO:nano-graphrag:Inserting 1 vectors to entities


...

llm output is {
    "title": "First Contact and Its Implications",
    "summary": "The community revolves around 'Alex', a leader coordinating preparations for potential first contact with an unknown intelligence, while navigating relationships that influence decision-making and responses. The dynamics include Alex's interactions with other characters and organizations.",
    "rating": 7.0,
    "rating_explanation": "The impact severity rating is moderate due to the complex nature of human interactions with potentially advanced extraterrestrial entities.",
    "findings": [
        {
            "summary": "Leadership Role of 'Alex'",
            "explanation": "As a central figure, Alex plays a pivotal role in coordinating preparations for first contact with an unknown intelligence, influencing decisions and guiding humanity's response. This highlights the criticality of leadership dynamics within the community."
        },
        {
            "summary": "Diverse Relationships",
   

INFO:nano-graphrag:Writing graph with 1220 nodes, 554 edges



Inserting page 67 from scraped_pages.json.gz | length: 38003
Chunk length: 3758


INFO:nano-graphrag:[New Docs] inserting 1 docs
INFO:nano-graphrag:[New Chunks] inserting 10 chunks
INFO:nano-graphrag:[Entity Extraction]...


Backup updated: /content/drive/MyDrive/erica/nano_graphrag_cache_ollama/backup
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
WE ARE INSIDE THE ollama_model_if_cache FUNCTION~~~~~~~~~~~~~~~~~~~~~~~
llm output is ("entity"<|>"Engineering AI Agents"<|>"organization"<|>"The organization developing a framework for Reinforcement Learning, possibly creating resources related to AI control theory.")## (